In [ ]:
! pip install numpy

In [ ]:
import numpy as np
from evedesign.system import System, Protein, SystemInstance, EntityInstance
from evedesign.models.boltzfold import BoltzFoldTransformer

In [ ]:
p = Protein(rep='TSENPLLALREKISALDEKLLALLAERRELAVEVGKAKLLSHRPVRDIDRERDLLERLITLGKAHHLDAHYITRLFQLIIEDSVLTQQALLQQH', id='EcCM', first_index=2)
s = System([p])
ei = EntityInstance(rep=np.array(list('TSENPLLALREKISALDEKLLALLAERRELAVEVGKAKLLSHRPVRDIDRERDLLERLITLGKAHHLDAHYITRLFQLIIEDSVLTQQALLQQH'), dtype='U1'))
inst = SystemInstance([ei])

m = BoltzFoldTransformer(
    device='cpu',
    sampling_steps=2,
    recycling_steps=1,
    use_msa_server=True, 
    diffusion_samples=5
)
m.build(s)
print(m.ready)
print(m.can_model(s))

In [ ]:
import tempfile, yaml
from pathlib import Path
from evedesign.models.boltz.convert import system_instance_to_yaml

with tempfile.TemporaryDirectory() as d:
    path = Path(d) / 'test.yaml'
    system_instance_to_yaml(s, inst, path, use_msa=False)
    data = yaml.safe_load(path.read_text())
    print(data)

In [ ]:
import logging
from loguru import logger
logger.enable("evedesign")

result = m.transform([inst])
print(result)

In [ ]:
result

In [ ]:
# Run process_inputs manually to check it works independently
import tempfile, shutil
from pathlib import Path
from boltz.main import process_inputs
from evedesign.models.boltz.convert import system_instance_to_yaml

tmp_dir = Path(tempfile.mkdtemp(prefix="boltzfold_test_"))
try:
    yaml_path = tmp_dir / "instance_0" / "input.yaml"
    system_instance_to_yaml(s, inst, yaml_path, use_msa=False)
    print(f"YAML written to {yaml_path}")
    print(yaml_path.read_text())
    
    import os
    cache = Path(os.environ.get("BOLTZ_CACHE", "~/.boltz")).expanduser()
    process_inputs(
        data=[yaml_path],
        out_dir=tmp_dir,
        ccd_path=cache / "ccd.pkl",
        mol_dir=cache / "mols",
        use_msa_server=False,
    )
    print("process_inputs completed")
    print("processed dir contents:")
    for f in (tmp_dir / "processed").rglob("*"):
        print(f)
finally:
    shutil.rmtree(tmp_dir, ignore_errors=True) 